In [ ]:
json_paths = ['data_by_year/all_articles_by_institution_cited_1992.json',
              'data_by_year/all_articles_by_institution_cited_1993.json',
              'data_by_year/all_articles_by_institution_cited_1994.json',
              'data_by_year/all_articles_by_institution_cited_1995.json',
              'data_by_year/all_articles_by_institution_cited_1996.json',
              'data_by_year/all_articles_by_institution_cited_1997.json',
              'data_by_year/all_articles_by_institution_cited_1998.json',
              'data_by_year/all_articles_by_institution_cited_1999.json',
              'data_by_year/all_articles_by_institution_cited_2000.json',
              'data_by_year/all_articles_by_institution_cited_2001.json',
              'data_by_year/all_articles_by_institution_cited_2002.json',
              'data_by_year/all_articles_by_institution_cited_2003.json',
              'data_by_year/all_articles_by_institution_cited_2004.json',
              'data_by_year/all_articles_by_institution_cited_2005.json',
              'data_by_year/all_articles_by_institution_cited_2006.json',
              'data_by_year/all_articles_by_institution_cited_2007.json',
              'data_by_year/all_articles_by_institution_cited_2008.json',
              'data_by_year/all_articles_by_institution_cited_2009.json',
              'data_by_year/all_articles_by_institution_cited_2010.json',
              'data_by_year/all_articles_by_institution_cited_2011.json',
              'data_by_year/all_articles_by_institution_cited_2012.json',
              'data_by_year/all_articles_by_institution_cited_2013.json']

In [ ]:
import pandas as pd
from collections import Counter
import json

def read_json(file_path):
    """Read JSON data from a file."""
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def extract_affiliation_data(affil, counter, geo_data):
    """Extract institution name, city, and country and update counters & geo_data."""
    if isinstance(affil, list):  # If multiple affiliations exist
        for aff in affil:
            if isinstance(aff, dict) and "affilname" in aff:
                inst_name = aff["affilname"]
                counter[inst_name] += 1
                geo_data[inst_name] = {
                    "city": aff.get("affiliation-city", "Unknown"),
                    "country": aff.get("affiliation-country", "Unknown"),
                }
    elif isinstance(affil, dict) and "affilname" in affil:
        inst_name = affil["affilname"]
        counter[inst_name] += 1
        geo_data[inst_name] = {
            "city": affil.get("affiliation-city", "Unknown"),
            "country": affil.get("affiliation-country", "Unknown"),
        }

def count_citations_and_references(json_paths):
    """Counts citations and references and extracts geo data."""
    reference_counter = Counter()
    cited_counter = Counter()
    geo_data = {}  # Stores institution name -> {city, country}

    for file in json_paths:
        data = read_json(file)
        for item in data:
            # Process cited by articles
            for cited in item.get("citedby_articles", []):
                if "affiliation" in cited:
                    extract_affiliation_data(cited["affiliation"], cited_counter, geo_data)

            # Process references
            for ref in item.get("references", []):
                if "affiliation" in ref:
                    extract_affiliation_data(ref["affiliation"], reference_counter, geo_data)

    return reference_counter, cited_counter, geo_data

def create_dataframe(json_paths):
    """Creates a Pandas DataFrame with institution name, counts, and location."""
    reference_counter, cited_counter, geo_data = count_citations_and_references(json_paths)

    # Get all unique institutions
    institutions = set(reference_counter.keys()) | set(cited_counter.keys())

    # Build the DataFrame
    data = []
    for inst in institutions:
        data.append({
            "name": inst,
            "reference_count": reference_counter.get(inst, 0),
            "cited_count": cited_counter.get(inst, 0),
            "city": geo_data.get(inst, {}).get("city", "Unknown"),
            "country": geo_data.get(inst, {}).get("country", "Unknown"),
        })

    return pd.DataFrame(data)

# Example usage

df = create_dataframe(json_paths)

# Print the resulting DataFrame
# print(df)

In [ ]:
import numpy as np

df["rate"] = df["reference_count"] / df["cited_count"].replace(0, np.nan)
#df["rate"].fillna(5000, inplace=True)
df["rate"] = round(df["rate"]).astype(int)

df["sum"] = df["reference_count"] + df["cited_count"]

print(np.min(df["rate"]), np.max(df["rate"]))

In [ ]:
df.to_csv("output_1999.csv", index=False)